# E0: reproduce FLOCK zero-shot results

- **Kaggle:** select the **GPU T4** accelerator, turn **Internet on**, choose **Save Version**, then
  **Save & Run All**. Download the `e0_*` files from the Output tab.
- **Colab:** choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Keep the tab
  open. The last cell downloads `e0_results.zip`.

The notebook checks the walker, runs three small inductive graphs for comparison with Table 9, and
times 20 Hetionet triples at the paper's zero-shot setting (n = 512, P = 2).

In [ ]:
import base64, importlib.util, io, json, os, subprocess, tarfile
from pathlib import Path

EXPERIMENT_COMMIT = "4d1fd030f26beeb76ea7cd90357265f210871648"
UPSTREAM_COMMIT = "f35103d25a78bdf4075de5c673a51de4979aa4d7"
WORK = Path(os.environ.get("E0_WORK", "/tmp/e0"))
ON_COLAB = importlib.util.find_spec("google.colab") is not None and "KAGGLE_KERNEL_RUN_TYPE" not in os.environ
OUTPUT = Path(os.environ.get("E0_OUTPUT", "/content/e0_output" if ON_COLAB else "/kaggle/working"))
REPO = WORK / "flock"
PYTHON = Path(os.environ.get("E0_PYTHON", WORK / "env/bin/python"))
SCRIPTS = REPO / "experiments/e0_reproduction"
RUN_ENV = {
    **os.environ,
    "EXPERIMENT_COMMIT": EXPERIMENT_COMMIT,
    "CUDA_VISIBLE_DEVICES": "0",
    "TQDM_DISABLE": "1",
}
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def sh(command, check=True, cwd=None):
    print("$", command, flush=True)
    bash_command = f"set -o pipefail; {command}"
    code = subprocess.run(bash_command, shell=True, executable="/bin/bash", cwd=cwd, env=RUN_ENV).returncode
    if code and check:
        raise RuntimeError(f"command failed with exit code {code}")
    if code:
        print(f"FAILED with exit code {code}; continuing", flush=True)
    return code == 0

In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout
print(gpu)
assert "T4" in gpu, "Select the GPU T4 accelerator in the notebook settings."
sh("nproc")

In [ ]:
if not REPO.exists():
    sh(f"git clone -q https://github.com/jw9730/flock {REPO}")
sh(f"git -C {REPO} checkout -q {UPSTREAM_COMMIT}")
BUNDLE = "H4sIAAAAAAAAA+19+3PbyJVufuZfgeLWvUslFAfvhyvcuo4tz3hje1weT7JbWi0GIiERI5JgCNC2kp352/ec7gbwNUBKoMae5FbEclkE0N2nH6fP6zsNbpJP8fUyv0yW8SJN5un2N5//Y9LH933xlz7tv/TQ/Y3lOaZlWa5j2nTfcj16fv0F+tL57Ioy2RLJbZ6Xd5W773l7cP+ffDzbmOWrVboup+7cupqbjnll+5dpehn4aRLM5pHpeIHte1e2ZYaB5bvh4O/d58fP5/uknzbpNuP1L776UjR4PwSBd3j/0/fW/ndcM/iN96U6hJ9/8v2P65+a8TbdbPP5blZm+fqz8cMD1t/3LPNx/X+Fz13rP1uks5v4Y7K8SbeTze2DaUj97x5Yf4vUi9Naf9vyLCr/Gcd58PNPvv7D4fBZvtok29QoF6mxTZdpUqRzg1edrmb5dp6tr42PWbkwnm426XqefTKeTuyxkaznhuAQqpiURpGmc1WvoIqbNCkng8G73bpQDSdL44cXy3x2MxFlfjDytXhyvU02CyO/EhebhNjxXwvjRXa9oy5ZRKYUD/6abvPTYpGXxjcpM2fKFMuS+jYZvKfn86RYEPlltiby2Vr26Uq2woP7y47YPC3GRpFze7fi5pqao6GXFXXRlYnx1Hj29nvjcpct5+rJIF1/yLb5mveJbJ1GRAxhvDt7+vz1mZEVRrrOd9cLGvL3RXKdPjE2t+WCRtjaQ8b56SnPlGGZtnvBV/mu3OxKYzL5apsWu6XchrJ8LCpPfizy9cWAFmowyFabnLqbbK+p20VaXS9o8MvssrrkCtX3vKi+bZZJeZVvV9V1sbukvT5Li7pEcVsMrrb5iuak5PYMdf8tXda0f8zWPyZ2dbXerWhQSWGsN9WtMt/OFtpFfJ3mq7TcZrPq9m2yWkpSrQKTeVImFd3n9J1Y6Oztt8ZUdGIUx1fZMo3jkwlNVr78kI5OJsy7JL3O7YvBs2/fvHj5NRUeFttZTHez8varWb4mPviKGYj5Jy63yboQIu5D+tXas2wqWKSry2VqT7hfQ6L46un7l9+++Y5aOh8us5t0ODaG86yovl4RK63nf6Y9MbwYvPn2+Zks+WonH3+TrL/Llzl/PVsRP+db/vqf+Tzhv8+J4RZ/YleLr54t0o+XyWwmHv17cnkpvrxKs4SaPnv+tWx6YNBnVBOo+lEROhm3CkBfqx50ysAgqt7d1Q50u1OsKtMMpi4CcwGNyYHWhWCWDhfS560qJGaqLqPNZrvIxWAwmKdXSirEViy2++jkiahM++sPYsc3MimRootZkmSNMVsmRZEWxjx/YqTJbGFcJTPeVrvCSIyrZUbSkSRiTvtBSMvydkNCwvid8T/v/mfCu5epsI9d8IoKrplk63n6acQ3Twzam+Lx2IjpH0sZsfwXol6ZZMtOPb4p68WiEl+361EnuJ7YZpOSOD3fjs5rBlcNkdRPWKjWjVU3tI6ciBbT+XUai2qdZsXgZDcKVXqblrvtWuzkkbihNzGVDcyScnTe3B1DiQlP7Mg8uRgb82w1tdRS183wALEVMeCxGvfvjGW6HtWjPbmA2iS54nU+T4splxHT2npaTUIx1YZpG79tNavqnSj+WubJPF5R08uKt0iwTlim0nQX6bYcmaSGyu1ISLavUFgNT+S0CdF4JZSlaKioRKJQoAO5tOmKRXpKyyBl8uS9ujOqWpbykIUl9ahMP5Uj1b6UiVSTBd6kSK7SmHs9qtqkGjT525Fi/elQ6mOLttP1ZldMz01aj3STzxbFlAZzSWtAf2Y3Gyq6JuVcjUMtv+j1SBI9H4oRDS9OJumHhKZIzVqREOl0JB6OxZ4bq+3A+nL6hlqVX2Ohc6YvkmWRqvnNrsQTVsOs0rnsk3op5eKtkvUuWcZcbMT/yf6tNxPSBvN8NakfVI1JMgbNfSoalBXkzhYtrnMWIPNqjWG4YgzSyhnJgXT3ycVJNfB5dp0W5UjaWqoxVujpltZHafZJsUhszx8p7qBNKtvizSkrNp2QdSe7zZxZQZabiMGQsizzy9syLUb6+qgqi/ST6kzVt0W+ick+LrOizGaF6uNYGFO39f6YSknbyNHXZEKy3bXIP3ILhbC1ZN10PjHe5kXG9QyLF6y4kYLzMp0lO5pr3tWnbJOXtTV5RSZnVtYyVNwcK5kXq70+NtbJilZNXcwzoic2LwtNNQlX1F/6ytvPPqmmZO9sSuG5bkQAtaLveTV98jE9JU662i2XoxFIE6P5ThenlqyUXCckBkpZR1gld1SaC/l2mefLZunlruDhtkQ+8ABv+Gwun9MXItZROGOjo0uAkeXAztsNXVBL9UN1b1yR44dt1cLdbNpVY99b8/12l0rJttmmH7J8R8s4223ZuKOnYtHPn4wN68mpRcKnvrafyMViVoX1qDvZaawp3qxE1a8DhSveZY3H/IV0s4IU1YeUrHFBVRX8tylyT1WQTJObj8mW22kYVDVmTKeGVUmzkTaa3xvmySRZM7PS8rOEG3Wr/555OlkuNXmUZNQt8sFK8vDPtmRijYZX+Y4ct4QnQDpJSmgmRkHO1FJuwMohqtwwaRANT+T6JMt8zcrjZ5hEqedosXERcBTcT146Hmf7vqXmUtQX3AqbWQ5Q1Wzdd1RFKZCErJm2OKHLkrr00gTh3+qpG3Jjwye0t0ptMSZF9tcUDIVhteR1iZgWmuaO/QzBFvOYFeITgzfxaFRzyP/RxYvebbWSSEfslDKPxdZRnWu62+pKTHKCGY3KkBVBdtHo54ZPz8X6XRj/1/gZeFLdPZms0mStkW63rZqpazZE9tHoReInGKgYIA1UDPjegVad6QwY+vLz/gH/3HvE3ZH+vLf5fq3jYEmvkyuySsrZgnZfnKgIS5zYDa0RipgpSq4TojnS5Ir+eA/xYUliNq7FR8yjpKYTcn9jtjrF3oo/5dumTM272JH/xsGetAl32ZeMvyKd7djpjjUxEcNG0x5Mit2qbxtyGtk4kw8SMkJvVyTISZrUE6mky7lW9WLPFOkCQnQwJtvh+joFDqjY47yRPRc9pE1rSX6qrF+2OEW4pzhkAiu5XkfoyK5R1to9prOorFw7YeIk6+t0ZJ8omSvjdke1phnirLcPN98VrDLiIxrI10ueGo4VMgtUQyPvgmeyviR9365ORrskLxtq2pDDUS2oC62+vCcibEUszWqspi/LKi0TnoERLADfv87K0W+T7fVOBM67HkATW5tsd+vR+ZAqkO/UVCELZvZxPmU/jb4lm5IjEjIUKCaUDC5y19RXwRdynidFOadi9GdLbrHSx/9icAT0j8SgSxHTTGm73hi79Ya2J4cwSMPPaDUN3uik52c0rWtjt6E20mRFl6SfRDR3I+MbWUlWeb7bzlIBDZPlLX1G7gXNHDuywqpGv3VsDIWJcCojl8ML5WZydSqbFxMVQZ1cp+VoePYfb8/evXx99uZ9/Ozb169fvh+eVKaPqkN91t24ui2eexLOH05FCFSEwc6ePh82RqbW0YkUqaPhHUDH8JANIGkSe8gvmiiap+zjZ1cZUSLLaZbGdWkhLUU32XfaFdzH01Py32ckBbK1vGR20HqqiSDmN8WWugAkD5ruS8cen2x2ZHSwn1/EyQfSoCzQqSA7EzT5BVGaxzT1ydVVtqYFG5kaORmspvJVkHgi78Qs7GmGRiftHUhlpVMbV2XiuLNL67BuVRoCvfvriT1Npck16hSoNyXNYWXlCh5gR7kKiU+eqi32VjwZzdNiRltF+KlxPM9n1BrLqhW7gttYxPSmdeV3ycfnTYVv0uXmRVX0BOhNkvk8rvbyaChD+kPpiU5JkY1ZSCS7ZTnlMP+dNeWeVxxID3hnqaLiDxcuqo0uZDK7ohBhkjKJo+bTTmSzDjm2/D8ZuK24nvUYFbhH7nM/JvuFv2pHoBfUEGygSn7SitaitG4IeYrFRiy9cBbKG8G7WR0WIcE8Ebe7ljeZTVR2b6CivaFk0XtUbbt7PymXlO0TBlYmcxo4UxDDHRs8o+tyap/UEky0INe1EV8CvoAnJ5OP26xMZVTuznaN3xnD/2IZNRhQ43HMLlAcs3obxjHvhTgeSjpyYxyXnnMX/ltdpDGHKWJGTx4EA9+D/7q+67fxX8fyrUf891f4DIfDd7u1jv1KnS716CYnxgf0VaC+CjAmdSzioe9Z2RiRhuEKIDSVAEptaqhw8lgJMm6rJJNTWie79WzBpiPtQRllxS6pgDUbmQJ6kUjt4IfT01Xy6ZSNoWVa/GAUtO1ThmNkVJftMEHv4yKjSmRT7Ir0areUMb9sxcj25a4U8Q85KDESHFQN6EoxIEHd/RvDOD1VAyyMP7+xwnfvXq4Vzvjkg8VP0w8ZG1WbHV1css93ytEEwzL+q5YTBwDhho4Adx8IBi/z62sac3UpJ6m+SqXRh2hwheKSgLgfGT6IBYuaCIRX1Sud8CWg4G/Onv3x7bcv37wXQUkJtDQcXXwl0BVlvk425WI4ePnm+ffP3r/801ncC0fO1jWIbPk1hmz5CkR+/+7pm++OavB+YPr90z+8Oouj+M3T1wIXlpp2+OIPlndjOwGyGyk6un9KX6QaG3Y5kov8+Q0UeXP26lW7wJtXUKBKveAH9ffx4KfB+29fndF4n51Rp8yJaSlLrdr4nwEHqxiBvP9lH4isLkL+V/WEZc0ATX1ubCwrjNXTxhtPyfyk7U77Bb2/Nlx0EEeSD+4CnxRquZsnWCDm0IksBHCi5JWR7LAM/fP2iFkrsbXJgmVsCJkSiwjlk9ouFAKQjF5lTavgKu+VUsAwo+GTYWWiktybGvs4lyyPCgZnw6NefImPtXdOhTGycaMbhKoRYqCqa80zRhaZs8jYGwqCUlwyPZ4kRYtchOVyCNUkDEkVTbh5KSzITllGKNkEJKZrBIRm6BFdNVHoeoqRnA/VkyHDFeq7HOoVh8N5cSa4WoqveVrHVSNT9fekqjhREzER+URT0bd6aZtSJByy9YR5NG5WmYo3F8jYVGMs6NYpD0tyYTjCKD0D8bxmDtEqXx1OhSDVfrPOP65FzkNhpJ/oz1II+x9og5EZuiYhevuDMc9J/QrtukiLOnWiqLE7dvB5NElxcz5kPhyKQM2wFjsv11cpCfFZOuwEVfQkAv7sTySohzPBlILmLtl75AnHzcM9uQV16538Ar1xfnyobX520WqzyTpo6tT3mqL9kicqzjkwrH2ZEoeriJEMuv1UNdgDa3W0Ru7JyJK+Yz0msvCST7EyyxRXratrRtEOL4b0886ti4pfoKUqKsQwGN4WaJu66LBNTUs2qGWusNAmY3U1qqufnD+BllWIVFiRWrdF0Ew52PJxdyR3j5PhK8l9hxoRCUSH2uCH51C/CTxSM2phhCEef0iWu7SQQKycnJJNW14G4XeyxCrqlBEtTiYbSNbJ8rbIqktZW9ihwz2pJdv8I6sYvsNfWRbwX06aFBXPhxHteVpXuns+nBdSAGiWzTn3VU8g+ttwtd1yiJgrvRjSjh0usrKwzPreYnhRBYg4r2THyQ9SUbLXLVyOXcOMUpFKG1AFAYQnMq3KiVJsL4vokjKcOXj5StwbDVNTac22sdAEECrhPVbuS6wU7CGNLmqCWpfXSreLC1DwKhYtVaTUPcwc8g4L+CYyJJlXFBFJrbG6qdSAqDGVf5RlUrEcA9Vq/9sXkzIfYSmhEGKpEKa9VUy7lWqxRAhqnyQRA0dxosfCYA1lOCcmylNWMuLqpCksdTNHiRgAm5UjKQNE4lNjDbDo2sTUqMptIW9tSA5kml0vykIgFir1qElk0vojSbUGWUWF2iZNO0tJGIPkz9BCbtKEhpWu8u2t6HMdBBS5MSwXMrbi0u0VsdKObIrtqE+KkvStKhksV0ltgHolGoNSboCxWtxUcs4UFn6sNmis2oWAYJESi4tkxz09NU7lOAYd1VD34j7VwOk2MVv3U6x/hG5ZyTQlZrxKvEhsT43lXNwkDQ6yplWA7/8/enBxosKELCFFBLcteQ8gDI09LETAnlA/iI5WvLSIOWJfc7pIOOOwLNrGyoGMN1pJMUVypxVoNNfShUq35A3G9rm2mnCa17gZQ5+pP9RQve48F9VdKFwtNy+C+ooAhtRUfL/KqWA0g+9CqWrFRThafsWnWwa9r2r7sy6jGIF4VjSoLnE1Mhb9cZkvU7IoRN3kshjdXf/E+D3pvdpxJhFRc3SddKgFsMV2EgFs8a37LBbqWcxcU4x0+r7pFOJFYZSVmLm+FCCsDAOxNtnuOPQ1qm+9+/67p1+fxd+dvXpBmn9HZv+nbcEUGOb47W/tdvvk0Wltax7vp+oRrRpLW/KNT+qmnIMisz05/8CoUOX9DBs8aG+sZtyNzYw7sZi7KYlJGjJUnNOXYnqu9JacswvogLhxZ2NNZHI/pHVnZQjHQu0vBJ7xBJ+yrVRVfytc7ar+PpOWq3x1c333DChQDhrtjc/tpBZrIl/K67jK2TboYmD148km34wqsd/cPoegZZWucNFNDG7sF2mRV1m+Wk5v06bil4vaERJ7srEfBcg0Mk/u2YWCydroH4J+TI2YkGQaC67zC6kl2SdgAuwU1DavCBMUmyXD5uOhllIhmj6XrVxUgP6dRn6TDHAIuKuaOxfZpDXYxj7BrliAGSMm7QCoV/GEevAPAukd9bnD3ftsNB5y/te1rcfzv7/C5xh3/6E07sZ/fddxu+d/Az94xH9/hc/fBsOIRfOAvgjrklEj2xuOB0MhjPlObYcYJGi3/Oh7DrdPnDDk7wtx4bv0/b28Hzn8Xd336PsL8dU1ucwLdd93Bz+NNbKe2Y+sA2Q912noOm5D13ODmrDj2Q1hz/fbhIN+43VNB8ZrNgN2TQsGbFnNiK0QRmzbbcKW2W/IrhvhTNsNZccHyg7MNQ7Z9x2N8p//2HuJLR/m2mkIO6bXEHYjvyZsw4jdyGrT7bnGlg90HbuZavlA0XUsp6ZriZlQhO0gbBPuucaO4Nd6wEEzYD8E5gLCTojM5XQI911jS+wgRdkO/WbIIVC2o6gZchg0lB1T309vXp323E6CnaoR27CdQlhiz41gxBbwlum36fblrQin2o8awkEAhEMg7How1VGbbk/ecs0A6DYrLMVTPd6wIesDa3m+16bbm7U0uQWs5YH48OxmEzsWTLRndxa4t/gILBAfkpp8EKL4CJpdrIuPSBdcr99bRpl86kXatoGrldBQgwauVuQU6wNpJ3Q6pBdpsuRD632oRzDlTgAqKvCROqiKEFjMDa0WddvIt9e9SJvI3ZbXcLcZwWpbXjPnsNaW3x62bRSzrN+YPRyzB2O2QS27PrIZymwvbJF2yMcpe5IGNeWajbawQ9hartsoSMeEUbstzcykyYPcJr2I+xYQD7yG03wP5FgQNWvth2CPhHZ7rd3+Ux4Ak0tdrcYd4YwfGnV7rd1jWNzHDRaAwvJRipL7BcNuiLf05GvOEE9W/QbtaNYf8JkLDB5aYJMEIEcjx2tZQxzy6ifOQthavgfWUICGZzPfngkiJWgbYSK/aS9VIImCxPca5exZwFx+2IgxECN+1KVp30/Ts0AxN+sqFUS1ro0d4jlAkhihQ9LpM0xNUTRr6mkWrt+oZA9VlB+21/T0g9uDagjGnmJb1R1czzCABcWxmrp6qnLa7qHqu7CkgbTu1HLBrg2iRlj5Eeyc0HQ6VHssKpniSBWkVAD6MAiRKuyaIOoQ7bGsUttV8ilouFf6J7XJA46LD6alb4Ydqj2WVTpC1VBB9cuZr4ZqNyzsew4M1e4YeH2WNUDhEFmNORuaoPLDKASquKwtacxU++xV28exwm71kZn8hqoXohhuSQim2mNdpfCpqTaC0EPjKnBBSEQo+7tD7bNbA9SzFhB1kCjYF56HuzXUlfw3rw3r5n6qJpoWZtRQNQO7oWrZzQSbPozVsqI2VacPVQdcBRMkk4nLalkRUA2Qatim6vWi6iJVC6jiWC0g6iLRzgQTlew677G0QFdtUnlfizK4oNFt2LB+y/d9+ertM+O7VSJSMO+TUCboOhdjOiYarR4YrSYsr9tyBgXpV4xB30/aRpsVIg2OCfaTC4ylm8utQbMRE0W6Q/geErvRDcXQCgzZRY727Ya33AjjZ57XIWy3eHo/YdvRrPRmmTVrFexk24FVdqKWMnjO741Itz0Iy0hJRThqCEsjohqxCSPW/F+d8LN8PUs35Zu0JDe4z7hNJG/5jQix0MxwgL1tV3OMnBb952ef9vD3gdUOgLjvBwdWu5l0F8dO5vUe2l0GPzDvaKm7GGhxMIjnY6AFDGa3Fan9z6dff+tYZp9Be2hOWo0F4JrAar4No7aQxQOd8NNtVpT5n9welC1caxlRqIJpwGoyylbJE1xrV1/r53/YpPMs6cloDjoLHppbNspRs1HLGNei6dDld3PQ4t6d7YG6cgLwQwPc2hBrccG2dAKrZb1b3s2p7QQjyzzpQx0Nage2t6RS7a/I2U/eNdvh8Yq83Y887jClseQD1NVgC9mBgyvehiMq6l4v6g664dIarKIuWnAPQh8WKDCPLYXBBd0QaWYPhGMQnnAg1HYQj0G268IivfEYHLrTrPsdeAwYSV7HU+0LxwBV4LbDYIyNw3XaVPtjMZqVBHpbx2Jgd/s4zR2AoO/y2hZOMywvam1gb/RWvVaU/ggkRkMlGjl6EIexYW3tDtW+sXKUoGANHgJhNFb2vA7Z/iBMiMM9HoTpgCE99xBaoR5E6A+CMDq0aHV85Z5cpYMhEcitQyCMLjXszoD7ojAWLnGEtgGOGC3+CK1B324T7staGKz1+qAwLtgknvtwFAYD8zL2XKEwaAJCwEkGxirH2dIpH4PCOKAZXMAinAgX2Wx8HNdC3diKThwJwujWJ0z4QQwG/RwSaQ/GYCwTltoC7z2CzSw9AbWZTfTeg/aE9wdhHIx9ubifHY3LwP7UQEbba5HuD8I4phbsa4SnHaIdhCAMrnUHcDsKg0EDNDQRjAAWh+BmgF586LShp/4QjIO5GW4ARoiWqAABemmx1fox6pA+gsUDZDQZgXyvHsC45QM1cIybt6K6x6AwUVTPuDUxa7pRs7Hl7Rfqdu3X0e2H4y/o1vkAJ2MM2wfR7Tk4WquT+dMnsOuZmAMDEdaDCIyLwU677dz0jOviwjYjPQjCaBFsrzvQPlFdZCaJF9wHwmDSjR91DPleYd0IQRjUTgdBGFzTB2IwGKwPomaoAe7aAIL1gQXWR2hZHap9MBgtgg1ush9quxWkFOKHkdch2gdaQ43guRgU0Cw8sGm1cF8rKtAbg0H+bcIgPk6v3ewZ38dQvWu1bZ1eCIymBEAUh5hgE4INHfgwv1ErztcXgQmQlUAoeQHsmkBGytXcANWwa7n32qsanAdJBzpVrxmrb/pI9WEQjI8M7ICEQBs2CEAWBtpY9XXtC8FgohiAEgcBmAAzWlrR094ATEPShByafvBLl2Yv+MUGTjIBGDYDF6ki1IQj7RDtDb84GCN2muEexl80J9DRKR+Hv2geQi/8BROlWpruGPxFBl7ruB2YysDL8n5luAHhtojqDb/o5gQkdJjaXANgizaM384C7I++YITBhSgO0vWcAwm1LbJHgC+uhY6+jQPW4KaGtzCfxHPb4MeR4IuD2s8GDxRDhCCc9ZTtNvJzFPKC6wwRHRdz0rTcRx9BTXcf7d7Ii3vIK8EUCBccohbyoiv9/siLhzoY7VbMzVJmjapwMEXqCNhFHy9Gk1wtXRxziWGyPfuXwC5SStbp+RCY9SDK4YO2cDHJpUX7CNjFiTB4hwHwCLnbAe5GmKu9qY/GXTzMPrSaxbY9lN0Q3bERdvmlqIuW+tjIMxtz8TDaYGPWZRu4Px52wRxyiE87Nkg1Dyxcx0LswWqDD4q8RvrtNhWvX+JjqjVhLZkbD79gYNzzEFNFTnfa/hKfAL6XKiJs0rSvfGCUZYB5YOaN3wqLS1H2mnbYbnX/eBFdU96vGi+6L4hvoX/oslMq8S1+rdyD8C3pe9ZT3nQh1JxFu9lmYYTOotNmtb6hah9dY+lnKC8Vwx0g1DC1K+wc+ukZp/Y1FBlcixBZOwBYLTKBw8IOoNk7UI04YgB7KkIoM2zESaS5b90TMD3XN9R0VqM7QoTsQ0gXjDzNbYzadHsub+gfCEKEiDaFYO/LiagmurXC/UGuCEMCIfBVZGGqYugcmOnWCh8Bc4W4lULI4Ysw6SmEEFOERlHkdNIVe+4k9HA8E+MRGrAHoQHUlF7QAV96yw5NXDeEAxetX6uRHYEN0UPf6aBNfWWHRrexewM8lOCDuR8i8hJ0YyE9eSvA4yc+5EsG4SGh5Wihy044pC9vBWgDomYM0eAO4Mhg5CAIELYj8f1hrkjL/QUREgUw25EHOxlB8uiX4VyBBtBDBCpCUygCDDdCRDPqIC/9ga4IrbAI4KYID3hFEKhukX440hVhbnmEo/ZdHLUFpD2c8/bZsv5IV4g4RAApICGC5iFAuRFCuSTrfgHUJUVxNW5Ix0CbPzKBdhjhjD8Y6YoQzA0jnHBca7cRZZI7Kspee9RHIV2hluhuNQItclBzRLjF7uDxI5AuC6GYBumy8Tidiek+QNe22+k+vcEuzeZshFmgp/I1y6yL0T3HcHodTUGD0z4Ai3govFFNBp3R9oNFtPO4DcAWWA4SbWii+xz4XXytR/xcg2I8iDvqehmYWUuJCDs0+2AiWkJC2IiNADeQD4lyIWYKBlH3DE6PNQ0xOgEnUYNIM0BAQiM+HXrdIzg9ljT00JQH6wNlswc4gU60/dqAfkhXoJ+vBkbC4JMHZ2Gk7KgNgAedNsIzZCEEf1pDPUSzi/702ae4pupAs3yAEScZTq+4F8NsThdf67GoAeIwePxRd0fBpIzQL2yfG+gHdPkIOXkuCkI0ZK2GlUJEavcNtc9W1XJZQmBg1LKYMhXi6dawpeGPQUSk6q4YCtI8IwuwpxD3DgZbola+wTGISKhBmVEz6hBHHcLJkAgPKkRm59RRH3jPR83uwusJVAqCegAuko+pj16bZh94z4u0nG1A4nHXuuB66/6g2znV1Qfg83FpXdhAWpYSHvkJMIXXix6M8PlosXqQiBd6GNMB1RN5eGbafjDahTEdF06Jeyg0MAlPOxzvPfiwUUtugLL1UNkC4VDLFWqleR4BePn66zYaZ1A/XgWiw9Yst1942sjCpTbDRn5gvBTP3ASRFqvdAzv1RbxgxuHki/LN0GGpHBMMKgVdun3RrhB3cwhH6SKEGEMHlDAaVq28mv5gl2Yuu6j7Q81yBYMDUxOiFrZ5BN6lQS+2iWnboCRsPAsbgcx0W6lax+Fd2puKIDkisLTMbVBQeOIkbBnsR+BdvnbgA9Y5xGNd0gys9BMQPoT49EO79EgexOIjBAHAD4wwRyIM2slxx8Fduo8EFl+IQHaAESY9SN32zI6DuwINxAffP8Rc6hCssAjtocitURj+feEHojC4+JCdeBCFibQc3+5xny+HwmCU/qEwTIBb3AdwV4NhkG6Eqb1hNwe0d5AeT08GEDs+jMOgAep08JC+OEyE4ICDYhzDahC/1XjM66ASvxiH0QAgYCw80NXe2Z8DhsGQFvgY+jy7D37jW4QRrRDc8xb+A6o6GOub+TODMBhdQqNbS30yO1hIT8YK8G0Kvg2Oq3Yc2Gw4K8CDgqR2HgjChJhiFsCLGwPM6PMBaGvF0zrO64NQGMj/wfO4fgA2KDpyQRfu6i07tFdVBgdC4wEcF2zFQ7pISG8QBszPCPLII1TQKC61AHEnMH4UBBPhMboQDgncAcFgbDxsUz8CgkF7PwK9GIUoRYKwP+n+EEyApAERAC6LUHB+LgQmwlBfCLhTiIMObTSHgLL1S973FuErQiMX1xpGDS8WjPBFSlHnWNkRCAyaBBEkM0boWmqkMcst8rvgzzE8jmZBhAiMjvVFh5a79Z67IyAYLenJgrQBG51aE20/TWNFD37lm5Z9BMCuHrJ3wPgL0Og0u+BEn/Au2vpWHxRGe9FR90B9v8MpSNXtAcNgFI5M/4fBMOhXwOvtDsMw2vx2p7dPcBcjQ4eWFFEY7XWNZvc8TB8UBg/vBx5aApi2h8YeJkuGe96/1geGQYc58HGsyEkBmra4qP6D3vp2GIfxtLECVU04d89W9YJhNEwek/TQ1PJggjEvMoy6h3/6HDnCd396YOB5oAEdyEEIUSC3d2pPHKYVYIRgLr7OCBdVcwwf9MY3PdQF1lWg5Ti7iDhp75nrgj89FhUT81wQ+3omIoYUtUNO7p6Xc/VFYWwMKWJGz2EUBlWtteeVZF8OhdFJdxCRL4PCBIgQeO2o3hfDYTA7rp06/+VgGMQkos5Q+8IwAU4y5lGH+EKsAN7YGyEAFLVP7fWGYfC0IFoUB1GYALMBvVaa1hEojBY0hfMgehoihC1DX7OfHnzsKNB+MgFsKIxh4RvAQnwBd9jaREejMBKQ3pfrYeH5AM86RH/PO996wzCI04OZrufFgQeO8crI/wVATMsta0ibmNDsYlzpc+Aw0YHEtFDLlYL3MuvhLN1y/RwojIvv+XDwnbp4NoGM7C+BwmBUGgFdRBnblt0RIIweTAN4M0KlHMKBvgiPloWdd1J/aRhGy7w8dOLpl6Iw+Ka5AGMt+I7fyDn0orueKIytHcOBAxJ4WDZEEEhj9HbWWt8zR1o0MTAREECLBOFV1Fpe22PqdeQosrQgNagOBFd9ONym48n+HiC755mjCK36EAI8joZ/oDzREl1tBrsGP/29f2LpH/qDv/+1Tf+yy7apuDi93uwm5ac7f/Kq7+fu3/8yLcvz2r//RraX//j7X7/C51+Mr99+b6TrD9k2X/PKi19nLBdkLzecMRY/Kp3vSuPt7Xv+vcix/NnI02ImfgXVSNZzUYd/gph/L/rjIpstjHWazo1n3z9/elps0ll2lc0M8fvjxWQwS7cl3ZhObdP2J5ZFanswW/APapana/5x1WX213Q7nToTb+IMZst8N99ks5tlyrcsupUmxS3/iPd0ak0sd3CVLVP+IfDp1J3YE7oumKRqPpqYg2y+Trgu0fkxW/+Y2LIhf/BjfrnMLrkZn4qtku3NblMkV4IQGQ2D1WaVlAt+TteDdbL9uEiWBbU8sbldUtYf8+3NJy4eDNa71eaWyxJVd7BJZjcJ/248lfapqU3Bv8E+nQbURXuwub3M1nPLkh0x6Zp/UFSUdogWFb+9TVZUnPpFjfHmTItSUHZokINilt1k5ekyTbZrJhlNLL4n6VsBX6XlblPmOXc3CAWR4lZ10HLpqlzwj4VuqMSsXDLZkG+Klb1Oc/kr10xPlP3LfMWzG5jUcnm7oY6epp/KdF3wjy3zE4sncLdd0nQ6XIsb+7hIU2qZbW+6+nsz++Pn8fP4efw8fh4/j5/Hz+Pn8fP4efw8fh4/j5/Hz+Pn8fNP9vlfKD6bKwDIAAA="
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as archive:
    archive.extractall(REPO)
print("code ready at", REPO)

In [ ]:
sh("pip install -q uv")
sh(f"uv venv -q --python 3.12 {WORK / 'env'}")
sh(f"uv pip install -q --python {PYTHON} --index-url https://download.pytorch.org/whl/cu126 torch==2.8.0")
sh(f"uv pip install -q --python {PYTHON} -r {REPO}/experiments/requirements-gpu.txt")
sh(
    f"uv pip install -q --python {PYTHON} --only-binary :all: torch-scatter==2.1.2 "
    "-f https://data.pyg.org/whl/torch-2.8.0+cu126.html"
)
sh(f"uv pip install -q --python {PYTHON} --no-deps .", cwd=REPO / "graph-walker")
sh(f"{PYTHON} -c 'import torch, graph_walker; print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=WORK)
sh(f"uv pip freeze --python {PYTHON} > {OUTPUT}/e0_gpu_freeze.txt")

In [ ]:
sh(
    f"{PYTHON} check_walker.py --output {OUTPUT}/e0_walker_check_kaggle.json "
    f"2>&1 | tee {OUTPUT}/e0_walker_check_kaggle.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 8 --data-root {WORK}/data "
    "--datasets FB15k237Inductive:v1,WN18RRInductive:v1,NELLInductive:v1 "
    f"--output {OUTPUT}/e0_zero_shot.json 2>&1 | tee {OUTPUT}/e0_zero_shot.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 1 --max-triples 20 --data-root {WORK}/data "
    f"--datasets Hetionet --output {OUTPUT}/e0_hetionet_timing.json 2>&1 | tee {OUTPUT}/e0_hetionet_timing.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
for name in ("e0_zero_shot.json", "e0_hetionet_timing.json"):
    path = OUTPUT / name
    if not path.exists():
        print(name, "is missing; check its log")
        continue
    for run in json.loads(path.read_text())["runs"]:
        print(
            run["dataset"], "| MRR", round(run["measured"]["mrr"], 3), "| paper", run["paper_all_relations"]["mrr"],
            "| full run", run["full_run"], "| s/triple", round(run["seconds_per_triple"], 2),
            "| peak GPU GB", round(run["peak_gpu_memory_gb"], 2),
        )

In [ ]:
if ON_COLAB:
    import shutil

    from google.colab import files

    files.download(shutil.make_archive("/content/e0_results", "zip", OUTPUT))